# mLLMCelltype: Cell Type Annotation Using Multiple LLMs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cafferychen777/mLLMCelltype/blob/main/mLLMCelltype_Colab_Template.ipynb)
[![Paper](https://img.shields.io/badge/Paper-bioRxiv-green)](https://www.biorxiv.org/content/10.1101/2025.04.10.647852v1)
[![GitHub](https://img.shields.io/badge/GitHub-mLLMCelltype-blue)](https://github.com/cafferychen777/mLLMCelltype)
[![PyPI](https://img.shields.io/pypi/v/mllmcelltype)](https://pypi.org/project/mllmcelltype/)

This notebook provides a complete guide to using **mLLMCelltype** for automated cell type annotation in single-cell RNA-seq data. The tool leverages multiple large language models (LLMs) to achieve accurate, consensus-based annotations.

## Overview
- **Multi-model consensus**: Supports multiple LLM providers including OpenAI, Anthropic, Google, and others
- **Uncertainty quantification**: Consensus Proportion and Shannon Entropy
- **Scanpy integration**: Compatible with existing Scanpy workflows
- **Cost optimization**: Consensus optimization reduces redundant API calls
- **Free models available via OpenRouter**

## 1. Installation

First, let's install mLLMCelltype and its dependencies:

In [ ]:
# Install mLLMCelltype with all optional dependencies
!pip install -q mllmcelltype[all]

# For Colab compatibility
!pip install -q ipywidgets

print("Installation complete.")

# Verify installation
import mllmcelltype
print(f"mLLMCelltype version: {mllmcelltype.__version__}")

## 2. API Key Configuration

mLLMCelltype supports multiple LLM providers. You need at least one API key to get started.

**Choose your preferred option:**
1. **Option A**: Use free models via OpenRouter (recommended for testing)
2. **Option B**: Use your own API keys for better performance
3. **Option C**: Upload a `.env` file with all your keys

In [ ]:
import os
import pandas as pd
from getpass import getpass

# Try Google Colab imports, fall back to standard if not in Colab
try:
    from google.colab import userdata, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Google Colab. Using standard input methods.")

# Option A: Quick start with OpenRouter (includes free models)
# Get your API key from: https://openrouter.ai/keys
if IN_COLAB:
    try:
        # Try to get from Colab secrets first
        os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
        print("OpenRouter API key loaded from Colab secrets.")
    except:
        # Manual input with getpass for security
        openrouter_key = getpass("Enter your OpenRouter API key (or press Enter to skip): ")
        if openrouter_key:
            os.environ['OPENROUTER_API_KEY'] = openrouter_key
            print("OpenRouter API key set.")
else:
    openrouter_key = getpass("Enter your OpenRouter API key (or press Enter to skip): ")
    if openrouter_key:
        os.environ['OPENROUTER_API_KEY'] = openrouter_key
        print("OpenRouter API key set.")

# Option B: Set individual API keys (uncomment the ones you have)
# os.environ['OPENAI_API_KEY'] = getpass("OpenAI API key: ")
# os.environ['ANTHROPIC_API_KEY'] = getpass("Anthropic API key: ")
# os.environ['GOOGLE_API_KEY'] = getpass("Google API key: ")
# os.environ['DEEPSEEK_API_KEY'] = getpass("DeepSeek API key: ")
# os.environ['QWEN_API_KEY'] = getpass("Qwen API key: ")

# Option C: Upload .env file (only in Colab)
if IN_COLAB:
    upload_env = input("Upload .env file? (y/n): ")
    if upload_env.lower() == 'y':
        uploaded = files.upload()
        if '.env' in uploaded:
            with open('.env', 'r') as f:
                for line in f:
                    line = line.strip()
                    if '=' in line and not line.startswith('#'):
                        key, value = line.split('=', 1)
                        os.environ[key.strip()] = value.strip().strip('"\'')
            print("Environment variables loaded from .env file.")

# Display available providers
available_providers = []
provider_map = {
    'OPENAI': 'openai',
    'ANTHROPIC': 'anthropic', 
    'GOOGLE': 'gemini',
    'OPENROUTER': 'openrouter',
    'DEEPSEEK': 'deepseek',
    'QWEN': 'qwen',
    'ZHIPU': 'zhipu',
    'GROK': 'grok',
    'MINIMAX': 'minimax',
    'STEPFUN': 'stepfun'
}

for env_key, provider_name in provider_map.items():
    if f'{env_key}_API_KEY' in os.environ and os.environ[f'{env_key}_API_KEY']:
        available_providers.append(provider_name)

print(f"\nAvailable providers: {available_providers}")
if not available_providers:
    print("No API keys found. Please set at least one API key above.")

## 3. Load Your Data

mLLMCelltype accepts marker genes in various formats. Choose the option that matches your data:

In [ ]:
# Option 1: Use example data (PBMC markers)
use_example = input("Use example PBMC data? (y/n): ")

if use_example.lower() == 'y':
    # Example PBMC marker genes
    marker_genes = {
        "Cluster_0": ["IL7R", "CD3D", "CD3E", "CD3G", "TRAC"],  # T cells
        "Cluster_1": ["CD79A", "MS4A1", "CD19", "BANK1"],      # B cells
        "Cluster_2": ["CD14", "LYZ", "S100A8", "S100A9"],      # Monocytes
        "Cluster_3": ["GNLY", "NKG7", "PRF1", "GZMB"],         # NK cells
        "Cluster_4": ["FCER1A", "CST3", "CLEC10A"],             # Dendritic cells
        "Cluster_5": ["FCGR3A", "MS4A7", "IFITM3"],             # CD16+ Monocytes
        "Cluster_6": ["PPBP", "PF4", "GP9"],                    # Platelets
    }
    species = "human"
    tissue = "PBMC"
    print("Loaded example PBMC marker genes.")
    
else:
    # Option 2: Upload your own file
    if IN_COLAB:
        print("Upload your marker genes file (CSV/TSV/Excel)")
        print("Expected format:")
        print("- Column 1: Cluster ID")
        print("- Columns 2+: Marker genes")
        
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
    else:
        filename = input("Enter path to marker genes file (CSV/TSV/Excel): ")
    
    # Read the file
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)
    elif filename.endswith('.tsv'):
        df = pd.read_csv(filename, sep='\t')
    elif filename.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(filename)
    else:
        raise ValueError("Unsupported file format. Please use CSV, TSV, or Excel.")
    
    # Convert to dictionary format
    marker_genes = {}
    for idx, row in df.iterrows():
        cluster_id = str(row.iloc[0])
        genes = [str(g) for g in row.iloc[1:].dropna() if str(g).strip()]
        if genes:
            marker_genes[cluster_id] = genes
    
    # Get species and tissue info
    species = input("Species (e.g., human, mouse): ") or "human"
    tissue = input("Tissue type (e.g., PBMC, brain, optional): ") or ""
    
    print(f"Loaded {len(marker_genes)} clusters.")

# Display loaded data
print("\nMarker genes preview:")
for i, (cluster, genes) in enumerate(marker_genes.items()):
    if i < 3:
        print(f"{cluster}: {', '.join(genes[:5])}{'...' if len(genes) > 5 else ''}")
if len(marker_genes) > 3:
    print(f"... and {len(marker_genes) - 3} more clusters")

## 4. Choose Models and Annotation Strategy

Select which models to use and whether to use single-model or multi-model consensus annotation:

In [ ]:
# Model selection based on available API keys
model_options = {
    'openai': [
        {'provider': 'openai', 'model': 'gpt-5'},
        {'provider': 'openai', 'model': 'gpt-4-turbo'},
        {'provider': 'openai', 'model': 'gpt-3.5-turbo'}
    ],
    'anthropic': [
        {'provider': 'anthropic', 'model': 'claude-sonnet-4-5-20250929'},
        {'provider': 'anthropic', 'model': 'claude-opus-4-5-20251101'},
        {'provider': 'anthropic', 'model': 'claude-haiku-4-5-20251001'}
    ],
    'gemini': [
        {'provider': 'gemini', 'model': 'gemini-3-pro'},
        {'provider': 'gemini', 'model': 'gemini-3-flash'}
    ],
    'deepseek': [
        {'provider': 'deepseek', 'model': 'deepseek-chat'}
    ],
    'openrouter': [
        # Premium models
        {'provider': 'openrouter', 'model': 'anthropic/claude-sonnet-4.5'},
        {'provider': 'openrouter', 'model': 'google/gemini-3-pro'},
        # Free models
        {'provider': 'openrouter', 'model': 'meta-llama/llama-3.3-70b-instruct:free'},
        {'provider': 'openrouter', 'model': 'qwen/qwen-2.5-7b-instruct:free'},
        {'provider': 'openrouter', 'model': 'google/gemma-2-9b-it:free'}
    ]
}

# Show available models
print("Available models based on your API keys:\n")
available_models = []
for provider in available_providers:
    if provider in model_options:
        print(f"{provider.upper()}:")
        for model in model_options[provider]:
            model_name = model['model']
            is_free = ':free' in model_name
            print(f"  - {model_name} {'(free)' if is_free else ''}")
            available_models.append(model)
        print()

if not available_models:
    print("No models available. Setting up default free models...")
    available_models = [
        {'provider': 'openrouter', 'model': 'meta-llama/llama-3.3-70b-instruct:free'},
        {'provider': 'openrouter', 'model': 'qwen/qwen-2.5-7b-instruct:free'}
    ]

# Choose annotation strategy
print("\nChoose annotation strategy:")
print("1. Single model (fast, lower cost)")
print("2. Multi-model consensus (uses multiple models for cross-validation)")
print("3. Free models only (no cost)")

strategy = input("\nSelect strategy (1/2/3) [default: 2]: ") or '2'

if strategy == '1':
    # Single model
    print("\nSelect a model (enter number):")
    for i, model in enumerate(available_models):
        print(f"{i+1}. {model['provider']}/{model['model']}")
    model_idx = int(input("Model number: ")) - 1
    selected_models = [available_models[model_idx]]
    
elif strategy == '3':
    # Free models only
    selected_models = [m for m in available_models if ':free' in m['model']]
    if not selected_models:
        print("No free models available. Using default OpenRouter free models...")
        selected_models = [
            {'provider': 'openrouter', 'model': 'meta-llama/llama-3.3-70b-instruct:free'},
            {'provider': 'openrouter', 'model': 'qwen/qwen-2.5-7b-instruct:free'}
        ]
else:
    # Multi-model consensus (default)
    # Select best models from each provider
    selected_models = []
    for provider in available_providers:
        if provider in model_options and model_options[provider]:
            selected_models.append(model_options[provider][0])  # Best model from each provider
    
    # Ensure at least 2 models for consensus
    if len(selected_models) < 2:
        if available_models:
            selected_models = available_models[:3]  # Take up to 3 models
        else:
            selected_models = [
                {'provider': 'openrouter', 'model': 'meta-llama/llama-3.3-70b-instruct:free'},
                {'provider': 'openrouter', 'model': 'qwen/qwen-2.5-7b-instruct:free'}
            ]

print(f"\nSelected models:")
for model in selected_models:
    print(f"  - {model['provider']}/{model['model']}")

# API key mapping
api_keys = {}
for model in selected_models:
    provider = model['provider']
    if provider == 'gemini':
        api_keys[provider] = os.environ.get('GOOGLE_API_KEY', '')
    elif provider == 'zhipu':
        api_keys[provider] = os.environ.get('ZHIPU_API_KEY', '')
    else:
        api_keys[provider] = os.environ.get(f'{provider.upper()}_API_KEY', '')

## 5. Run Cell Type Annotation

Now let's perform the annotation. This section includes both single-model and multi-model consensus options.

### Demo Mode

If no API keys are configured, the notebook uses pre-computed results for the example PBMC dataset. This allows you to explore the workflow and see example outputs. Demo mode only works with the example PBMC data. For your own datasets, configure at least one API key (free options available via OpenRouter).

In [ ]:
from mllmcelltype import interactive_consensus_annotation, annotate_clusters
import os
import json

# Check if we have API keys and can run actual annotation
has_api_keys = any(
    os.environ.get(f'{provider.upper()}_API_KEY') 
    for provider in ['OPENAI', 'ANTHROPIC', 'GOOGLE', 'OPENROUTER', 'DEEPSEEK', 'QWEN']
)

# Check if using example PBMC data
using_example_data = use_example.lower() == 'y' if 'use_example' in globals() else False

# Determine if we should use demo mode
use_demo_mode = not has_api_keys and using_example_data

if use_demo_mode:
    print("Note: Using pre-computed demo results.")
    print("No API keys detected. Loading cached results for the PBMC example dataset.")
    print("These are cached results from a previous run, not live LLM predictions.")
    print("To run actual annotations, configure at least one API key above.\n")
    
    # Load cached results
    try:
        # Load the detailed results JSON
        with open('demo_data/cached_detailed_results.json', 'r') as f:
            formatted_results = json.load(f)
        
        print(f"Demo results loaded. Annotations for {len(formatted_results['consensus'])} clusters.")
        
    except FileNotFoundError:
        print("Error: Demo data files not found.")
        print("Please ensure 'demo_data/' directory exists with cached results.")
        raise
        
else:
    # Original annotation code - run actual LLM annotation
    print("Starting annotation with LLMs...\n")
    
    # Set up parameters
    annotation_params = {
        'marker_genes': marker_genes,
        'species': species,
        'tissue': tissue if tissue else None,
        'use_cache': True,  # Save API costs by caching results
        'verbose': True     # Show progress
    }
    
    if len(selected_models) == 1:
        # Single model annotation
        model = selected_models[0]
        print(f"Using single model: {model['provider']}/{model['model']}")
        
        # Use the correct parameter name
        results = annotate_clusters(
            **annotation_params,
            model=model,
            api_key=api_keys.get(model['provider'], '')
        )
        
        # Format results for consistency
        formatted_results = {
            'consensus': results,
            'consensus_proportion': {k: 1.0 for k in results.keys()},
            'entropy': {k: 0.0 for k in results.keys()},
            'model_annotations': {k: {model['model']: v} for k, v in results.items()},
            'controversial_clusters': []
        }
        
    else:
        # Multi-model consensus annotation
        print(f"Using {len(selected_models)} models for consensus annotation")
        
        # Advanced parameters for consensus
        consensus_params = {
            'consensus_threshold': 0.6,    # Minimum agreement (lowered for efficiency)
            'entropy_threshold': 1.2,      # Maximum entropy (raised for efficiency)
            'max_discussion_rounds': 3,    # Maximum discussion rounds
            'consensus_model': None        # Auto-select best model for consensus
        }
        
        # Show advanced options
        use_advanced = input("\nUse advanced consensus settings? (y/n) [default: n]: ") or 'n'
        if use_advanced.lower() == 'y':
            print("\nAdvanced settings (press Enter for defaults):")
            ct = input(f"Consensus threshold (default {consensus_params['consensus_threshold']}): ")
            if ct: 
                consensus_params['consensus_threshold'] = float(ct)
            
            et = input(f"Entropy threshold (default {consensus_params['entropy_threshold']}): ")
            if et: 
                consensus_params['entropy_threshold'] = float(et)
            
            mr = input(f"Max discussion rounds (default {consensus_params['max_discussion_rounds']}): ")
            if mr: 
                consensus_params['max_discussion_rounds'] = int(mr)
        
        # Run consensus annotation
        formatted_results = interactive_consensus_annotation(
            **annotation_params,
            models=selected_models,
            api_keys=api_keys,
            **consensus_params
        )
    
    print("\nAnnotation complete.")

# Summary regardless of mode
print(f"\nResults Summary:")
print(f"   - Annotated clusters: {len(formatted_results['consensus'])}")
print(f"   - Average consensus: {sum(formatted_results['consensus_proportion'].values()) / len(formatted_results['consensus_proportion']):.2%}")
if formatted_results.get('controversial_clusters'):
    print(f"   - Controversial clusters: {len(formatted_results['controversial_clusters'])}")

## 6. View and Analyze Results

Let's examine the annotation results and confidence metrics:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create results dataframe
results_df = pd.DataFrame({
    'Cluster': list(formatted_results['consensus'].keys()),
    'Cell Type': list(formatted_results['consensus'].values()),
    'Consensus Score': list(formatted_results.get('consensus_proportion', {}).values()),
    'Entropy': list(formatted_results.get('entropy', {}).values())
})

# Display results table
print("Cell Type Annotations:\n")
print(results_df.to_string(index=False))

# Identify high-confidence vs uncertain annotations
high_confidence = results_df[results_df['Consensus Score'] >= 0.8]
uncertain = results_df[results_df['Consensus Score'] < 0.8]

print(f"\nSummary:")
print(f"- Total clusters: {len(results_df)}")
print(f"- High confidence (>=80%): {len(high_confidence)}")
print(f"- Uncertain (<80%): {len(uncertain)}")

if len(uncertain) > 0:
    print(f"\nUncertain annotations:")
    print(uncertain[['Cluster', 'Cell Type', 'Consensus Score']].to_string(index=False))

# Visualization
if len(results_df) > 1 and len(selected_models) > 1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Consensus scores
    ax1.bar(results_df['Cluster'], results_df['Consensus Score'])
    ax1.axhline(y=0.8, color='r', linestyle='--', label='High confidence threshold')
    ax1.set_xlabel('Cluster')
    ax1.set_ylabel('Consensus Score')
    ax1.set_title('Annotation Confidence by Cluster')
    ax1.legend()
    if len(results_df) > 5:
        ax1.set_xticklabels(results_df['Cluster'], rotation=45)
    
    # Entropy distribution
    ax2.bar(results_df['Cluster'], results_df['Entropy'])
    ax2.set_xlabel('Cluster')
    ax2.set_ylabel('Shannon Entropy')
    ax2.set_title('Annotation Uncertainty (Entropy)')
    if len(results_df) > 5:
        ax2.set_xticklabels(results_df['Cluster'], rotation=45)
    
    plt.tight_layout()
    plt.show()

# Show controversial clusters if using consensus
if 'controversial_clusters' in formatted_results and formatted_results['controversial_clusters']:
    print(f"\nControversial clusters requiring discussion:")
    for cluster in formatted_results['controversial_clusters']:
        print(f"  - {cluster}")
        if 'model_annotations' in formatted_results and cluster in formatted_results['model_annotations']:
            print("    Model opinions:")
            for model, annotation in formatted_results['model_annotations'][cluster].items():
                print(f"      {model}: {annotation}")

## 7. Export Results

Save your results in various formats for downstream analysis:

In [ ]:
import json
from datetime import datetime

# Save as CSV
output_filename = 'mllmcelltype_results.csv'
results_df.to_csv(output_filename, index=False)
print(f"Results saved to {output_filename}")

# Save detailed results as JSON (includes all model opinions)
detailed_filename = 'mllmcelltype_detailed_results.json'
with open(detailed_filename, 'w') as f:
    json.dump(formatted_results, f, indent=2)
print(f"Detailed results saved to {detailed_filename}")

# Download files (only in Colab)
if IN_COLAB:
    files.download(output_filename)
    files.download(detailed_filename)

# Create a summary report
report = f"""mLLMCelltype Annotation Report
==============================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Species: {species}
Tissue: {tissue if tissue else 'Not specified'}
Total clusters: {len(results_df)}
Models used: {len(selected_models)}

High Confidence Annotations (>=80% consensus):
{high_confidence[['Cluster', 'Cell Type', 'Consensus Score']].to_string(index=False) if len(high_confidence) > 0 else 'None'}

Uncertain Annotations (<80% consensus):
{uncertain[['Cluster', 'Cell Type', 'Consensus Score']].to_string(index=False) if len(uncertain) > 0 else 'None'}

Models Used:
{", ".join([f"{m['provider']}/{m['model']}" for m in selected_models])}
"""

report_filename = 'mllmcelltype_report.txt'
with open(report_filename, 'w') as f:
    f.write(report)
    
print(f"\nReport saved to {report_filename}")
if IN_COLAB:
    files.download(report_filename)

## 8. Integration with Scanpy (Optional)

If you have an AnnData object from Scanpy, you can directly integrate the results:

In [ ]:
# This cell is optional - only run if you have Scanpy data
try:
    import scanpy as sc
    
    # Example: If you have an AnnData object
    # adata = sc.read_h5ad('your_data.h5ad')
    
    # Add annotations to AnnData
    def add_annotations_to_adata(adata, results_df, cluster_col='leiden'):
        """Add mLLMCelltype annotations to AnnData object"""
        # Create mapping dictionary
        cluster_to_celltype = dict(zip(results_df['Cluster'], results_df['Cell Type']))
        cluster_to_confidence = dict(zip(results_df['Cluster'], results_df['Consensus Score']))
        
        # Map to cells
        adata.obs['celltype_mllm'] = adata.obs[cluster_col].astype(str).map(cluster_to_celltype)
        adata.obs['celltype_confidence'] = adata.obs[cluster_col].astype(str).map(cluster_to_confidence)
        
        # Add to uns for reference
        adata.uns['mllmcelltype_results'] = results_df.to_dict()
        
        print(f"Added annotations to {len(adata.obs['celltype_mllm'].dropna())} cells.")
        
        return adata
    
    print("Scanpy integration function defined.")
    print("\nTo use with your AnnData:")
    print("adata = add_annotations_to_adata(adata, results_df, cluster_col='your_cluster_column')")
    
except ImportError:
    print("Scanpy not installed. Skip this section if not needed.")

## 9. Advanced Features

Explore additional capabilities of mLLMCelltype:

In [ ]:
# 1. Processing multiple datasets
print("Processing Multiple Datasets:")
print("""
# Process multiple datasets with a simple loop
datasets = {
    'sample1': marker_genes_1,
    'sample2': marker_genes_2,
    'sample3': marker_genes_3
}

results = {}
for name, markers in datasets.items():
    results[name] = annotate_clusters(
        marker_genes=markers,
        species='human',
        provider='openai',
        model='gpt-4-turbo'
    )
""")

# 2. Custom prompts for specialized contexts
print("\nCustom Prompt Example:")
print("""
# For specialized tissues or conditions
custom_prompt = '''You are analyzing {species} {tissue} from a patient with autoimmune disease.
Focus on immune cell subtypes and activation states.'''

results = annotate_clusters(
    marker_genes=marker_genes,
    custom_prompt=custom_prompt,
    species='human',
    tissue='synovial fluid',
    model=model,
    api_key=api_key
)
""")

# 3. Hierarchical annotation
print("\nHierarchical Annotation Example:")
print("""
# First level: broad cell types
level1_results = annotate_clusters(
    marker_genes, 
    species=species,
    model=model,
    api_key=api_key
)

# Second level: detailed subtypes for immune cells
immune_clusters = [c for c, ct in level1_results.items() 
                  if any(term in ct.lower() for term in ['immune', 't cell', 'b cell'])]
immune_markers = {k: marker_genes[k] for k in immune_clusters}

level2_results = annotate_clusters(
    immune_markers,
    species=species,
    tissue=tissue,
    custom_prompt='Focus on detailed immune cell subtypes and activation states.',
    model=model,
    api_key=api_key
)
""")

# 4. Cost estimation
print("\nCost Estimation:")
total_clusters = len(marker_genes)
models_used = len(selected_models)

print(f"Clusters to annotate: {total_clusters}")
print(f"Models used: {models_used}")
print(f"Total API calls: ~{total_clusters * models_used}")
print("\nAPI costs vary by provider. See provider documentation for current pricing.")
print("OpenRouter free models: $0")

# 5. Cache management
print("\nCache Management:")
print("""
from mllmcelltype import get_cache_stats, clear_cache

# Check cache statistics
stats = get_cache_stats()
print(f"Cache size: {stats['total_size_mb']:.2f} MB")
print(f"Cached results: {stats['total_entries']}")

# Clear cache if needed
# clear_cache()  # Uncomment to clear
""")

## 10. Resources and Next Steps

### Documentation and Support
- **Documentation**: [GitHub Wiki](https://github.com/cafferychen777/mLLMCelltype/wiki)
- **Paper**: [bioRxiv](https://www.biorxiv.org/content/10.1101/2025.04.10.647852v1)
- **Issues**: [GitHub Issues](https://github.com/cafferychen777/mLLMCelltype/issues)
- **Discussions**: [GitHub Discussions](https://github.com/cafferychen777/mLLMCelltype/discussions)

### Tips for Best Results
1. **Use 3-5 top marker genes** per cluster for best accuracy
2. **Provide tissue context** when available
3. **Use consensus annotation** for critical analyses
4. **Review uncertain annotations** (consensus < 80%)
5. **Cache results** to avoid redundant API calls

### Common Issues
- **Rate limits**: Use fewer models or add delays between calls
- **API errors**: Check your API keys and quotas
- **Uncertain results**: Try adding more specific marker genes
- **Import errors**: Make sure to install with `pip install mllmcelltype[all]`

### Citation
If you use mLLMCelltype in your research, please cite:
```
Yang, C., Zhang, X., & Chen, J. (2025). Large Language Model Consensus Substantially Improves 
the Cell Type Annotation Accuracy for scRNA-seq Data. bioRxiv. 
doi: https://doi.org/10.1101/2025.04.10.647852
```

## Summary

This tutorial covered the full mLLMCelltype workflow: installation, API configuration, data loading, single- and multi-model annotation, result visualization, export, Scanpy integration, and advanced features.

For further information, see the [documentation](https://github.com/cafferychen777/mLLMCelltype/wiki) and [paper](https://www.biorxiv.org/content/10.1101/2025.04.10.647852v1).